# UCS420: Cognitive Computing — Lab Assignment 9
## Part B: Emotion-aware Cognitive Assistant using Facial Expression Detection

**Name:** Divyanshi Dhawan**Roll No:** 1024160057

### 1. Problem and design
An assistant that only reads text cannot tell when a user is frustrated, anxious or sad. This system **perceives** the user's face through a camera (or photos/video), **understands** their emotional state, **reasons** over time (is the distress momentary or sustained?) and **adapts** its response.

```
 Camera / image / video ─► Face detection ─► Emotion classifier ─► Temporal smoothing ─► Policy / reasoning ─► Adaptive response
                           (OpenCV / DeepFace)  (7-class CNN)        (moving average)      (distress rules)        (message + action)
```

| Cognitive stage | Component |
|---|---|
| Perception | Face detector + pre-trained facial-emotion CNN (DeepFace) |
| Understanding | 7-class probability vector: angry, disgust, fear, happy, sad, surprise, neutral |
| Reasoning | Moving-average smoothing, sustained-distress rule with cool-down |
| Interaction | Empathetic message + suggested action (breathing break, notify caregiver — simulated) |

**Note:** the notebook uses a **pre-trained model** for the main pipeline and includes an **optional section** to train your own CNN on FER-2013.

## 2. Setup

In [ ]:
!pip -q install deepface tf-keras opencv-python-headless

In [ ]:
import os, glob, random, time
from collections import deque, Counter
import numpy as np, cv2, pandas as pd
import matplotlib.pyplot as plt
EMOTIONS = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]   # DeepFace order
NEGATIVE = {"angry", "disgust", "fear", "sad"}

## 3. Input: webcam, uploaded images or video
Use **any** of the following to supply faces (Colab runs in the cloud, so the browser webcam is captured through JavaScript).

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.9):
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => capture.onclick = resolve);
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth; canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    open(filename, 'wb').write(b64decode(data.split(',')[1]))
    return filename

os.makedirs('/content/faces', exist_ok=True)

# Option 1 — webcam: call this once per expression (happy, neutral, sad, ...)
# take_photo('/content/faces/shot1.jpg')

# Option 2 — upload images
# from google.colab import files
# for n, d in files.upload().items(): open(f'/content/faces/{n}', 'wb').write(d)

image_paths = sorted(sum([glob.glob(f'/content/faces/*.{e}') for e in ('jpg', 'jpeg', 'png')], []))
print(len(image_paths), "images ready:", [os.path.basename(p) for p in image_paths])

## 4. Perception — face detection + emotion recognition

In [ ]:
from deepface import DeepFace
DETECTOR = "opencv"      # alternatives: 'ssd', 'mtcnn', 'retinaface' (more accurate, slower)

def detect_emotions(img_bgr, detector=DETECTOR):
    """Return one dict per detected face: {box:(x,y,w,h), probs:{emotion: 0..1}, dominant:str}."""
    try:
        res = DeepFace.analyze(img_path=img_bgr, actions=['emotion'], detector_backend=detector,
                               enforce_detection=False, silent=True)
    except Exception as e:
        print("DeepFace error:", e); return []
    faces = []
    for r in (res if isinstance(res, list) else [res]):
        if r.get("face_confidence", 1) == 0: continue            # no real face found
        reg = r["region"]
        probs = {k: float(v) / 100.0 for k, v in r["emotion"].items()}
        faces.append({"box": (reg["x"], reg["y"], reg["w"], reg["h"]), "probs": probs, "dominant": r["dominant_emotion"]})
    return faces

def annotate(img_bgr, faces):
    out = img_bgr.copy()
    for f in faces:
        x, y, w, h = f["box"]
        cv2.rectangle(out, (x, y), (x + w, y + h), (0, 200, 0), 2)
        cv2.putText(out, f"{f['dominant']} {f['probs'][f['dominant']]:.0%}", (x, max(20, y - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 200, 0), 2)
    return out

In [ ]:
rows = []
for p in image_paths:
    img = cv2.imread(p)
    faces = detect_emotions(img)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    ax[0].imshow(annotate(img, faces)[:, :, ::-1]); ax[0].axis('off'); ax[0].set_title(os.path.basename(p))
    if faces:
        pr = faces[0]["probs"]; ax[1].bar(EMOTIONS, [pr[e] for e in EMOTIONS]); ax[1].set_ylim(0, 1)
        ax[1].set_title("Emotion probabilities"); ax[1].tick_params(axis='x', rotation=45)
        rows.append({"image": os.path.basename(p), "faces": len(faces), "dominant": faces[0]["dominant"],
                     "confidence": round(pr[faces[0]["dominant"]], 2)})
    else:
        ax[1].text(0.1, 0.5, "No face detected"); ax[1].axis('off')
        rows.append({"image": os.path.basename(p), "faces": 0, "dominant": None, "confidence": None})
    plt.tight_layout(); plt.show()
pd.DataFrame(rows)

## 5. Reasoning — the emotion-aware assistant (temporal policy)
A single frame is noisy (blinks, speech and lighting change the prediction). The assistant therefore:

1. **Smooths** the probability vector with a moving average over the last *N* observations.
2. Computes a **distress score** = smoothed probability of *angry + disgust + fear + sad*.
3. **Escalates** only if distress stays above a threshold for *k* consecutive observations (sustained distress), otherwise it responds to the dominant emotion.
4. Avoids repeating the same message every frame (only speaks when the emotion changes or escalation starts).

In [ ]:
RESPONSES = {
    "happy":    ("You look happy — that's great to see!", "Offer to log this moment / continue the task."),
    "neutral":  ("Everything seems calm. I'm here if you need anything.", "Continue normal interaction."),
    "surprise": ("That looked unexpected. Would you like me to explain what just happened?", "Offer a clear explanation."),
    "sad":      ("You seem a little down. Would you like to talk, or hear something uplifting?", "Offer supportive content / check-in."),
    "angry":    ("You seem frustrated. Let's slow down — I can simplify this step for you.", "Simplify the task, lower the pace."),
    "fear":     ("You seem worried. You're safe, and I'll guide you step by step.", "Give reassurance and step-by-step guidance."),
    "disgust":  ("Something seems off. Do you want me to change what I'm showing?", "Offer an alternative."),
}
ESCALATION = ("You've seemed upset for a while. Let's pause for a short breathing break. "
              "Would you like me to let someone you trust know?", "Start breathing exercise; notify caregiver (simulated).")

class EmotionAssistant:
    def __init__(self, window=5, distress_threshold=0.6, sustain=3):
        self.hist = deque(maxlen=window)
        self.thr, self.sustain = distress_threshold, sustain
        self.run, self.last = 0, None

    def update(self, probs):
        v = np.array([probs[e] for e in EMOTIONS], dtype=float)
        self.hist.append(v)
        sm = np.mean(self.hist, axis=0)
        emo = EMOTIONS[int(sm.argmax())]
        distress = float(sum(sm[EMOTIONS.index(e)] for e in NEGATIVE))
        self.run = self.run + 1 if distress >= self.thr else 0
        escalate = self.run == self.sustain                      # fire once when the condition is first met
        msg, action = ESCALATION if escalate else RESPONSES[emo]
        speak = escalate or emo != self.last                     # avoid repeating the same message
        self.last = emo
        return {"emotion": emo, "distress": round(distress, 3), "escalate": escalate,
                "speak": speak, "message": msg if speak else None, "action": action, "smoothed": sm}

### 5.1 Simulated session (shows the policy working, independent of any camera)

In [ ]:
rng = np.random.RandomState(7)
def fake_probs(dominant, strength=0.7):
    p = rng.dirichlet(np.ones(len(EMOTIONS)) * 0.6) * (1 - strength)
    p[EMOTIONS.index(dominant)] += strength
    return dict(zip(EMOTIONS, p / p.sum()))

script = ["neutral"]*4 + ["happy"]*4 + ["neutral"]*3 + ["sad"]*3 + ["fear"]*3 + ["sad"]*4 + ["neutral"]*4 + ["happy"]*3
assistant = EmotionAssistant(window=4, distress_threshold=0.55, sustain=3)
log = []
for t, emo in enumerate(script):
    out = assistant.update(fake_probs(emo))
    log.append({"t": t, "observed": emo, **{k: out[k] for k in ("emotion", "distress", "escalate", "message")}, "sm": out["smoothed"]})
    if out["message"]: print(f"t={t:2d}  [{emo:8s}] -> {out['emotion']:8s} distress={out['distress']:.2f}  |  {out['message']}")

sm = np.array([l["sm"] for l in log])
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].stackplot(range(len(sm)), sm.T, labels=EMOTIONS); ax[0].legend(ncol=7, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, 1.25))
ax[0].set_ylabel("smoothed probability")
ax[1].plot([l["distress"] for l in log], 'r-o'); ax[1].axhline(assistant.thr, ls='--', c='gray')
for l in log:
    if l["escalate"]: ax[1].axvline(l["t"], c='k', ls=':'); ax[1].text(l["t"] + .1, .9, "ESCALATE", rotation=90)
ax[1].set_ylabel("distress score"); ax[1].set_xlabel("time step"); plt.tight_layout(); plt.show()

### 5.2 Run the assistant on your photos / video frames

In [ ]:
assistant = EmotionAssistant(window=3, distress_threshold=0.55, sustain=2)
for p in image_paths:                               # images are treated as consecutive observations
    faces = detect_emotions(cv2.imread(p))
    if not faces:
        print(os.path.basename(p), "-> no face found, skipping"); continue
    out = assistant.update(faces[0]["probs"])
    print(f"{os.path.basename(p):20s} raw={faces[0]['dominant']:8s} smoothed={out['emotion']:8s} distress={out['distress']:.2f}")
    if out["message"]: print("   ASSISTANT:", out["message"], "| ACTION:", out["action"])

In [ ]:
def process_video(path, every_n=10, max_frames=300):
    """Sample every n-th frame of a video, run the assistant, return the log as a DataFrame."""
    cap, i, asst, rows = cv2.VideoCapture(path), 0, EmotionAssistant(window=5, sustain=3), []
    while cap.isOpened() and len(rows) < max_frames:
        ok, frame = cap.read()
        if not ok: break
        if i % every_n == 0:
            faces = detect_emotions(frame)
            if faces:
                o = asst.update(faces[0]["probs"])
                rows.append({"frame": i, "raw": faces[0]["dominant"], "smoothed": o["emotion"], "distress": o["distress"], "escalate": o["escalate"]})
        i += 1
    cap.release()
    return pd.DataFrame(rows)

# df = process_video('/content/faces/my_video.mp4'); display(df.head()); df.plot(x='frame', y='distress')

## 6. (Optional) Train your own emotion CNN on FER-2013
The pipeline above uses a pre-trained model. To show your own model-building, train a small CNN on the FER-2013 dataset (48×48 grey-scale faces, 7 emotions). **Use a GPU runtime** (Runtime → Change runtime type). You need a Kaggle account and `kaggle.json`; the folder-per-class version of FER-2013 on Kaggle (slug `msambare/fer2013` — verify it still exists) has `train/` and `test/` folders.

In [ ]:
# from google.colab import files; files.upload()                       # upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d msambare/fer2013 -p /content/fer --unzip

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def load_fer(root='/content/fer'):
    kw = dict(color_mode='grayscale', image_size=(48, 48), batch_size=64, label_mode='categorical')
    train = tf.keras.utils.image_dataset_from_directory(f'{root}/train', validation_split=0.1, subset='training', seed=1, **kw)
    val   = tf.keras.utils.image_dataset_from_directory(f'{root}/train', validation_split=0.1, subset='validation', seed=1, **kw)
    test  = tf.keras.utils.image_dataset_from_directory(f'{root}/test', shuffle=False, **kw)
    return train, val, test

def build_cnn(n_classes=7):
    m = models.Sequential([
        layers.Input((48, 48, 1)),
        layers.Rescaling(1/255.), layers.RandomFlip('horizontal'), layers.RandomRotation(0.08), layers.RandomZoom(0.1),
        layers.Conv2D(32, 3, padding='same', activation='relu'), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding='same', activation='relu'), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding='same', activation='relu'), layers.BatchNormalization(), layers.MaxPooling2D(),
        layers.Dropout(0.3), layers.Flatten(), layers.Dense(256, activation='relu'), layers.Dropout(0.5),
        layers.Dense(n_classes, activation='softmax')])
    m.compile('adam', 'categorical_crossentropy', metrics=['accuracy'])
    return m

# ---- uncomment to train ----
# train_ds, val_ds, test_ds = load_fer()
# class_names = train_ds.class_names                      # alphabetical folder names
# cnn = build_cnn(len(class_names))
# hist = cnn.fit(train_ds, validation_data=val_ds, epochs=25,
#                callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
# print("Test accuracy:", cnn.evaluate(test_ds)[1])

In [ ]:
# ---- evaluation: confusion matrix + curves (uncomment after training) ----
# from sklearn.metrics import confusion_matrix, classification_report
# y_true = np.concatenate([y.numpy().argmax(1) for _, y in test_ds]); y_pred = cnn.predict(test_ds).argmax(1)
# print(classification_report(y_true, y_pred, target_names=class_names))
# cm = confusion_matrix(y_true, y_pred, normalize='true')
# plt.figure(figsize=(6, 5)); plt.imshow(cm, cmap='Blues'); plt.xticks(range(7), class_names, rotation=45); plt.yticks(range(7), class_names)
# plt.colorbar(); plt.title("Normalised confusion matrix"); plt.show()
# plt.plot(hist.history['accuracy'], label='train'); plt.plot(hist.history['val_accuracy'], label='val'); plt.legend(); plt.show()

# ---- plug your CNN into the assistant (replaces DeepFace's classifier, keeps its face detector) ----
# def cnn_probs(img_bgr, box):
#     x, y, w, h = box
#     face = cv2.cvtColor(img_bgr[y:y+h, x:x+w], cv2.COLOR_BGR2GRAY)
#     face = cv2.resize(face, (48, 48))[None, :, :, None].astype('float32')
#     p = cnn.predict(face, verbose=0)[0]
#     return {name: float(p[i]) for i, name in enumerate(class_names)}      # keys match EMOTIONS names

## 7. Results, limitations and ethics (write your own observations)

* **Observed behaviour:** _which expressions were recognised reliably (usually happy, surprise) and which were confused (fear ↔ surprise, sad ↔ neutral, disgust is rare in training data)._
* **Effect of smoothing:** compare `raw` and `smoothed` columns; smoothing removes one-frame flickers but adds a small delay.
* **Limitations:** lighting, head pose, glasses/masks and low resolution reduce accuracy; FER-2013 labels are noisy (even human agreement is only about 65%); a face expression is **not** the same thing as a felt emotion, and expression norms differ across cultures.
* **Bias and privacy:** models can perform unevenly across age, gender and skin tone; always obtain **explicit consent**, process frames locally where possible, **do not store** face images, and let the user switch the camera off.
* **Responsible use:** the assistant only *offers* support (a break, a check-in); it never diagnoses a mental state or acts without the user's agreement. Real distress should be handled by a human.
* **Future work:** multimodal sensing (voice tone + text sentiment), per-user calibration, on-device inference, and a user study of how helpful the interventions are.